# Drug DDI GNN — Kaggle quickstart

This notebook runs the refactored DGL project in an isolated virtual environment.
Attach a Kaggle Dataset containing `drugIDandSMILES.csv` and `interaction_list.csv`.
Read `docs/KAGGLE.md` before installing: use Python 3.10–3.12 and a matching DGL/PyTorch wheel.


In [ ]:
import sys, torch
print('Python:', sys.version)
print('PyTorch in Kaggle kernel:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError('This DGL compatibility profile requires Python 3.10–3.12. Use a supported Kaggle runtime.')


## Install in a separate environment

The next cell shows the CUDA 12.4 installation. For CPU-only execution, use the alternate PyTorch CPU and DGL CPU wheel commands documented in `docs/KAGGLE.md`. If your Kaggle driver/runtime does not match CUDA 12.4, choose the matching official wheel rather than forcing this command.


In [ ]:
%cd /kaggle/working/drugddi-gnn
!python -m venv /kaggle/working/ddi-venv
VENV_PY = '/kaggle/working/ddi-venv/bin/python'
!$VENV_PY -m pip install --upgrade pip
!$VENV_PY -m pip install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu124
!$VENV_PY -m pip install dgl==2.4.0 -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!$VENV_PY -m pip install -r requirements.txt
!$VENV_PY -m pip install -e .


In [ ]:
VENV_PY = '/kaggle/working/ddi-venv/bin/python'
!$VENV_PY scripts/check_environment.py
!$VENV_PY -c "import torch, dgl; print('Torch:', torch.__version__, 'DGL:', dgl.__version__, 'CUDA:', torch.cuda.is_available())"


In [ ]:
DATA_DIR = '/kaggle/input/<your-data-dataset>'
!$VENV_PY scripts/train.py --data-dir $DATA_DIR --model graphsage --epochs 200 --device auto --output-dir /kaggle/working/artifacts/graphsage


## Predict candidate interactions

Prepare a CSV with `Drug1 ID` and `Drug2 ID` columns. Both DrugBank IDs must exist in the graph used for training.


In [ ]:
PAIRS_CSV = '/kaggle/input/<your-pairs-dataset>/candidate_pairs.csv'
!$VENV_PY scripts/predict.py --checkpoint /kaggle/working/artifacts/graphsage/model.pt --graph /kaggle/working/artifacts/graphsage/train_graph.bin --pairs-csv $PAIRS_CSV --output /kaggle/working/predictions.csv
